> **Learning-log notebook (kept as written).** Scratch work from building this project step by step.
> It predates the final cleanup, so some imports use the old module names (`Transformer_LM`, `training_together`, `AdamW.py`, `train_bpe_prallel`),
> and data paths point to local files that are not in this repo. Very large cell outputs were cleared. The code in `cs336_basics/` is the maintained version.


## BPE

### BPE Tokenizer - Reading

In [1]:
import regex as re
from collections import Counter

input_path = "../data/owt_train.txt"
vocab_size = 500
special_tokens = ["<|endoftext|>"]

In [2]:
# Vocab size sanity
# assert len(vocab) == 256 + len(special_tokens)

# Pair counting: hand-verify on a toy
# toy_pretokens = Counter({(108, 111, 119): 5, (108, 111): 2})  # "low"*5, "lo"*2
# Expected pair counts:
#   (108, 111) = 5 + 2 = 7    ← lo
#   (111, 119) = 5            ← ow
# Run your pair-counting code on `toy_pretokens` and confirm those numbers.

# Encoding sanity
# assert tuple("low".encode("utf-8")) == (108, 111, 119)

In [3]:
def get_all_pairs(tup):
    return [tup[i:i+2] for i in range(len(tup)-1)]
print(get_all_pairs((32, 116, 111)), get_all_pairs((258, 111)))


def get_token_pair_count(word_count):
    if len(word_count) == 0:
        return Counter()
    token_pairs_count = {}
    for word in word_count:
        if len(word) < 2:
            continue
        freq = word_count[word]
        for i in range(len(word)-1):
            pair = tuple(word[i:i+2])
            if pair in token_pairs_count:
                token_pairs_count[pair] += freq
            else:
                token_pairs_count[pair] = freq

    token_pairs_count = Counter(token_pairs_count)
    return token_pairs_count


# token_counts.most_common(1) with lex max
def get_top_pair_lex(token_pairs_count):
    if len(token_pairs_count)==0:
        return None
    max_val = max(token_pairs_count.values())
    top_pair = max([c for c, v in token_pairs_count.items() if v == max_val])
    return top_pair



def replace_pair_in_tup(tup, pair, new_id):
    new_tup = []
    replace = False
    for i in range(len(tup)-1):
        if replace:
            replace = False
            continue
        elif tup[i:i+2] == pair:
            new_tup.append(new_id)
            replace = True
        else:
            new_tup.append(tup[i])
    # if tup[-2:] != pair:
    if replace == False: # before the last one, the two in pair have been replaced, then append the last 
        new_tup.append(tup[-1])
    return tuple(new_tup)

print(replace_pair_in_tup((108, 108), (108, 108), 258))
assert replace_pair_in_tup((108, 108, 108), (108, 108), 258) == (258, 108)
assert replace_pair_in_tup((108, 108, 108, 108), (108, 108), 258) == (258, 258)

def get_keys_to_replace(word_count, top_pair, new_id):
    keys_to_replace = {}
    for key in word_count:
        if top_pair[0] not in key or top_pair[1] not in key or len(key) < 2:
            continue
        else:
            new_key = replace_pair_in_tup(key, top_pair, new_id)
            if new_key != key:
                keys_to_replace[key] = new_key
    return keys_to_replace






[(32, 116), (116, 111)] [(258, 111)]
(258,)


In [4]:
# def update_pair_count(token_pairs_count, top_pair, word_count, keys_to_replace, new_id):
#     del token_pairs_count[top_pair]
#     for old_key in keys_to_replace:
#         old_key_count = word_count[old_key]
#         new_key = keys_to_replace[old_key]
#         old_pairs = get_all_pairs(old_key) # [(32, 116), (116, 111)]
#         new_pairs = get_all_pairs(new_key) # [(258, 111)]
#         for i in range(len(old_pairs)):
#             pair = old_pairs[i]
#             if pair  == top_pair:
#                 continue
#             elif (pair[0] == top_pair[1]) and i >0 and old_pairs[i-1] == top_pair: # (23, 116, next), e.g. (116, 111)
#                 token_pairs_count[pair] -= old_key_count # old_key_count number of these pair disappear from the token_pairs_count
#             elif (pair[1] == top_pair[0]) and (i<len(old_pairs)-1) and old_pairs[i+1] == top_pair:# (prev, 32, 116) and 
#                 token_pairs_count[pair] -= old_key_count # old_key_count number of these pair disappear from the token_pairs_count
#         for pair in new_pairs:
#             if new_id in pair:
#                 if pair in token_pairs_count:
#                     token_pairs_count[pair] += old_key_count # new pair appear  this number of times
#                 else:
#                     token_pairs_count[pair] = old_key_count 


#         # update the byte_table

#         if new_key not in word_count:
#             word_count[new_key] = word_count[old_key]
#         else:
#             word_count[new_key] += word_count[old_key]
#         del word_count[old_key] 

#     return token_pairs_count, word_count

In [5]:
# token_pairs_count
def update_pair_count(token_pairs_count, top_pair, word_count, keys_to_replace, new_id):
    for old_key in keys_to_replace:
        old_key_count = word_count[old_key]
        new_key = keys_to_replace[old_key]
        old_pairs = get_all_pairs(old_key) # [(32, 116), (116, 111)]
        new_pairs = get_all_pairs(new_key) # [(258, 111)]
        old_pairs_counts = Counter(old_pairs) 
        delta = Counter(new_pairs)
        delta.subtract(old_pairs_counts)

        for pair, d in delta.items():
            token_pairs_count[pair] += d * old_key_count
        

        if new_key not in word_count:
            word_count[new_key] = word_count[old_key]
        else:
            word_count[new_key] += word_count[old_key]
        del word_count[old_key] 
    return token_pairs_count, word_count

In [43]:
# ============ TESTS for update_pair_count ============

def build_pair_counts(bt):
    """Recompute pair counts from scratch (the brute-force ground truth)."""
    pc = Counter()
    for k, v in bt.items():
        for i in range(len(k) - 1):
            pc[k[i:i+2]] += v
    return pc

def build_keys_to_replace(bt, top_pair, new_id):
    """Reproduce your keys_to_replace logic."""
    out = {}
    for k in bt:
        if top_pair[0] not in k or top_pair[1] not in k or len(k) < 2:
            continue
        nk = replace_pair_in_tup(k, top_pair, new_id)
        if nk != k:
            out[k] = nk
    return out

def run_one_merge(bt, top_pair, new_id):
    pc = build_pair_counts(bt)
    ktr = build_keys_to_replace(bt, top_pair, new_id)
    pc_new, bt_new = update_pair_count(pc, top_pair, bt, ktr, new_id)
    # filter zero/negative entries for clean comparisons
    pc_new = Counter({p: c for p, c in pc_new.items() if c > 0})
    return pc_new, bt_new

def assert_invariant(pc, bt, label=""):
    total = sum(v for v in pc.values() if v > 0)
    expected = sum((len(k) - 1) * v for k, v in bt.items() if v > 0)
    assert total == expected, f"[{label}] pair-sum {total} != expected {expected}"

def assert_matches_brute_force(pc, bt, label=""):
    """Incremental pc must equal a recount from scratch."""
    pc_brute = Counter({p: c for p, c in build_pair_counts(bt).items() if c > 0})
    pc_inc = Counter({p: c for p, c in pc.items() if c > 0})
    assert pc_inc == pc_brute, f"[{label}] mismatch.\n incr: {pc_inc}\n brute: {pc_brute}"


# Test 1: basic merge, no overlap
bt = Counter({(32, 116, 111): 5})
pc, bt = run_one_merge(bt.copy(), (32, 116), 258)
assert bt == Counter({(258, 111): 5}), bt
assert_invariant(pc, bt, "T1")
assert_matches_brute_force(pc, bt, "T1")

# Test 2: pair with both elements equal — (a, a) merge over "lll"
bt = Counter({(108, 108, 108): 3})
pc, bt = run_one_merge(bt.copy(), (108, 108), 258)
assert bt == Counter({(258, 108): 3}), bt
assert_invariant(pc, bt, "T2")
assert_matches_brute_force(pc, bt, "T2")

# Test 3: regression — shared-byte but non-touching pair must NOT be decremented
bt = Counter({(116, 116, 32, 116): 4})   # "tt t" x4
pc, bt = run_one_merge(bt.copy(), (32, 116), 258)
assert bt == Counter({(116, 116, 258): 4}), bt
assert pc[(116, 116)] == 4, pc[(116, 116)]   # preserved!
assert_invariant(pc, bt, "T3")
assert_matches_brute_force(pc, bt, "T3")

# Test 4: multiple non-overlapping merge sites in one pretoken
bt = Counter({(32, 116, 97, 32, 116): 2})
pc, bt = run_one_merge(bt.copy(), (32, 116), 258)
assert bt == Counter({(258, 97, 258): 2}), bt
assert_invariant(pc, bt, "T4")
assert_matches_brute_force(pc, bt, "T4")

# Test 5: adjacent same-merges "  t t" -> (258, 258)
bt = Counter({(32, 116, 32, 116): 1})
pc, bt = run_one_merge(bt.copy(), (32, 116), 258)
assert bt == Counter({(258, 258): 1}), bt
assert_invariant(pc, bt, "T5")
assert_matches_brute_force(pc, bt, "T5")

# Test 6: two pretokens collapse to the same new_key (tests accumulation + del of old)
bt = Counter({(32, 116, 97): 3, (258, 97): 10})
pc, bt = run_one_merge(bt.copy(), (32, 116), 258)
assert bt[(258, 97)] == 13, bt[(258, 97)]      # 10 + 3
assert (32, 116, 97) not in bt, "old_key must be deleted"
assert_invariant(pc, bt, "T6")
assert_matches_brute_force(pc, bt, "T6")

# Test 7: pretoken with no occurrence — must be untouched
bt = Counter({(97, 98, 99): 7})
pc, bt = run_one_merge(bt.copy(), (32, 116), 258)
assert bt == Counter({(97, 98, 99): 7}), bt
assert_invariant(pc, bt, "T7")
assert_matches_brute_force(pc, bt, "T7")

# Test 8: longer pretoken with two overlapping-byte but valid merges
bt = Counter({(32, 116, 116, 32, 116, 116): 1})  # " tt tt"
pc, bt = run_one_merge(bt.copy(), (32, 116), 258)
# Expected: (32,116)→258 at pos 0-1 and pos 3-4
#   → (258, 116, 258, 116)
assert bt == Counter({(258, 116, 258, 116): 1}), bt
assert_invariant(pc, bt, "T8")
assert_matches_brute_force(pc, bt, "T8")

print("All 8 tests passed ✓")
    

NameError: name 'replace_pair_in_tup' is not defined

In [7]:


def train_bpe(
        input_path,
        vocab_size = 500,
        special_tokens = ["<|endoftext|>"],
        **kwargs
):
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

    vocab = {n:bytes([n]) for n in range(256)}
    for t in range(len(special_tokens)):
        vocab[t+256] = special_tokens[t].encode('utf-8')
    with open(input_path, "r", encoding='utf-8') as f:
        lines = f.read()



    # this is the typical way of how to use several special tokens as hard boundaries and split the input string
    # into several parts -> here split by '<|endoftext|>‘ and it's not in the splitted input string
    sorted_specials = sorted(special_tokens, key=len, reverse=True)
    pattern = "|".join(re.escape(t) for t in sorted_specials)

    docs = re.split(pattern, lines)
    # further split each doc/paragraph into list of words by the GPT2 split pattern
    docs_splits = [re.findall(PAT, d) for d in docs]
    # take the first list of words in splitted docs as example
    # sample = docs_splits[0]


    # count word frequency
    word_table = Counter()
    for doc in docs_splits:
        word_table.update(doc)
    # convert to counter as 
    word_count = Counter({tuple(s.encode('utf-8')):word_table[s] for s in word_table})

    # merge all the keys with touch of this top pair, also add the merge to be a new word in the vocabluary
    token_pairs_count = get_token_pair_count(word_count)
    merges = []
    # for n in range(total_num_merged):
    while len(vocab) < vocab_size:
        top_pair = get_top_pair_lex(token_pairs_count)
        if top_pair is None or token_pairs_count[top_pair] <=0:
            break
        merges.append((vocab[top_pair[0]], vocab[top_pair[1]]))
        new_id = len(vocab)
        vocab[new_id] = vocab[top_pair[0]]+vocab[top_pair[1]]
        keys_to_replace = get_keys_to_replace(word_count, top_pair, new_id)
        token_pairs_count, word_count = update_pair_count(token_pairs_count, top_pair, word_count, keys_to_replace, new_id)
    return vocab, merges

In [8]:

def train_bpe_parallel(
        input_path,
        vocab_size = 500,
        special_tokens = ["<|endoftext|>"],
        **kwargs
):
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

    vocab = {n:bytes([n]) for n in range(256)}
    for t in range(len(special_tokens)):
        vocab[t+256] = special_tokens[t].encode('utf-8')
    with open(input_path, "r", encoding='utf-8') as f:
        lines = f.read()



    # this is the typical way of how to use several special tokens as hard boundaries and split the input string
    # into several parts -> here split by '<|endoftext|>‘ and it's not in the splitted input string
    sorted_specials = sorted(special_tokens, key=len, reverse=True)
    pattern = "|".join(re.escape(t) for t in sorted_specials)

    docs = re.split(pattern, lines)
    # further split each doc/paragraph into list of words by the GPT2 split pattern
    docs_splits = [re.findall(PAT, d) for d in docs]
    # take the first list of words in splitted docs as example
    # sample = docs_splits[0]


    # count word frequency
    word_table = Counter()
    for doc in docs_splits:
        word_table.update(doc)
    # convert to counter as 
    word_count = Counter({tuple(s.encode('utf-8')):word_table[s] for s in word_table})

    # merge all the keys with touch of this top pair, also add the merge to be a new word in the vocabluary
    token_pairs_count = get_token_pair_count(word_count)
    merges = []
    # for n in range(total_num_merged):
    while len(vocab) < vocab_size:
        top_pair = get_top_pair_lex(token_pairs_count)
        if top_pair is None or token_pairs_count[top_pair] <=0:
            break
        merges.append((vocab[top_pair[0]], vocab[top_pair[1]]))
        new_id = len(vocab)
        vocab[new_id] = vocab[top_pair[0]]+vocab[top_pair[1]]
        keys_to_replace = get_keys_to_replace(word_count, top_pair, new_id)
        token_pairs_count, word_count = update_pair_count(token_pairs_count, top_pair, word_count, keys_to_replace, new_id)
    return vocab, merges

In [44]:


import tempfile, os
with tempfile.NamedTemporaryFile('w', suffix='.txt', delete=False) as f:
    f.write(" the the the")
    p = f.name
p

'/var/folders/9d/97cx3pw53h3drj91bljrd4000000gn/T/tmpdorba8rl.txt'

In [ ]:
from train_bpe import train_bpe, train_bpe_prallel
# filepath = 'tests/fixtures/tinystories_sample_5M.txt'
filepath = '../data/TinyStoriesV2-GPT4-train.txt'
vocab_size = 10000
vocab, merges = train_bpe(filepath, vocab_size=vocab_size, special_tokens=["<|endoftext|>"])

print(merges)
# Expected: [(b't', b'h'), (b'th', b'e'), (b' ', b'the')]
# os.unlink(p)

[output cleared: 174 KB]


In [8]:
vocab_parallel, merges_parallel = train_bpe_prallel(filepath, vocab_size=vocab_size, special_tokens=["<|endoftext|>"])
print(merges == merges_parallel)
print(vocab == vocab_parallel)

True
True


In [25]:
from train_bpe import read_pkl, save_pkl
# vocab_tiny_story = 'cs336_basics/test/vocab_tiny_story.pkl'
# merges_tiny_story = 'cs336_basics/test/merges_tiny_story.pkl'
# save_pkl(vocab_parallel, vocab_tiny_story)
# save_pkl(merges, merges_tiny_story)


## owt
memory not enough

In [ ]:
filepath = '../data/owt_train.txt'
# vocab_parallel, merges_parallel = train_bpe_prallel(filepath, vocab_size=vocab_size, special_tokens=["<|endoftext|>"])
# vocab_owt = 'cs336_basics/test/owt_tiny_story.pkl'
# merges_owt = 'cs336_basics/test/owt_tiny_story.pkl'
# save_pkl(vocab_parallel, vocab_owt)
# save_pkl(merges, merges_owt)

## BPE Training on TinyStories

In [41]:
from multiprocessing import Pool
from pretokenization_example import find_chunk_boundaries
num_processes = 10

def get_chunk_word_count(chunk, special_tokens):
    sorted_specials = sorted(list(special_tokens), key=len, reverse=True)
    pattern = "|".join(re.escape(t) for t in sorted_specials)

    docs = re.split(pattern, chunk)
    # further split each doc/paragraph into list of words by the GPT2 split pattern
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

    docs_splits = [re.findall(PAT, d) for d in docs]
    # take the first list of words in splitted docs as example
    # sample = docs_splits[0]


    # count word frequency
    chunk_word_count = Counter()
    for doc in docs_splits:
        chunk_word_count.update(doc)

    return chunk_word_count

def pretokenize_chunk(args):
    input_path, special_tokens, start, end =  args
    with open(input_path, 'rb') as f:
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
    chunk_word_count = get_chunk_word_count(chunk, special_tokens)
    return chunk_word_count

#   ┌──────────────────────────────┬────────────────────────┬─────────────────────────────────────────┐
#   │             Mode             │ What you get from read │         What seek/tell count in         │
#   ├──────────────────────────────┼────────────────────────┼─────────────────────────────────────────┤
#   │ "rb" (binary)                │ bytes                  │ bytes                                   │
#   ├──────────────────────────────┼────────────────────────┼─────────────────────────────────────────┤
#   │ "r", encoding="utf-8" (text) │ str                    │ opaque cookies, NOT character positions │
#   └──────────────────────────────┴────────────────────────┴─────────────────────────────────────────┘

def parallel_pretokenize(input_path, num_processes = 10, special_tokens = b"<|endoftext|>"):

    with open(input_path, 'rb') as f:
        boundaries = find_chunk_boundaries(f, num_processes, special_tokens)
    with Pool(num_processes) as pool:
        per_chunk_count = pool.map(pretokenize_chunk, list(zip(
            [input_path]*(len(boundaries)-1), [special_tokens]*(len(boundaries)-1),
            boundaries[:-1], boundaries[1:]
            )))
    bytes_count = Counter()
    for chunk in per_chunk_count:
        bytes_count.update(chunk)
    return bytes_count

In [42]:
tiny_stories_sample = 'tests/fixtures/tinystories_sample.txt'
tiny_stories_5m = 'tests/fixtures/tinystories_sample_5M.txt'
tiny_stories_train = '../data/TinyStoriesV2-GPT4-train.txt'
from train_bpe_parallel import parallel_pretokenize
word_count_ts = parallel_pretokenize(tiny_stories_train)


In [9]:
for key, c in word_count_ts.most_common(10):
    print(bytes(key).decode('utf-8'))

.
,
 the
 and
 a


 to
 was
 They
 it


## Tokenizer


In [30]:
import regex as re
from collections import Counter
import pickle
from pathlib import Path
def save_pkl(param, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("wb") as f:
        pickle.dump(param, f)

def read_pkl(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("rb") as f:
        return pickle.load(f)
def split_special(lines, special_tokens, keep_split = False):
    if special_tokens is None:
        return [lines]
    sorted_specials = sorted(special_tokens, key =len, reverse = True)
    pattern = "|".join(re.escape(t) for t in sorted_specials)
    if keep_split:
        docs = re.split("("+pattern+")", lines)
    else:
        docs = re.split(pattern, lines)
    return docs
def ecd(s):
    return s.encode('utf-8')

def string_to_byte_tup(string):
    return tuple([bytes([i]) for i in ecd(string)])


def get_merge_pair(encoded_tup, merges_rank):
    # print('start merge', encoded_tup)
    min_pair, min_rank, merge_idx = None, float('inf'), None
    i = 0
    while i<len(encoded_tup):
        pair = encoded_tup[i:i+2]
        # print(pair)
        if pair in merges_rank:
            
            # print(pair, merges_rank[pair])
            cur_rank = merges_rank[pair]
            # print(pair, "mergable with rank:", cur_rank)
            if cur_rank < min_rank:
                min_pair = pair
                min_rank = cur_rank
                merge_idx = [i]
            elif cur_rank == min_rank:
                merge_idx.append(i)
        i+= 1
    # print("pair to merge:", min_pair)
    return min_pair, min_rank, merge_idx




def merge_single_idx(encoded_tup, idx):
    merged = encoded_tup[idx]+encoded_tup[idx+1]
    return encoded_tup[:idx]+(merged, )+encoded_tup[idx+2:]
def merge_tuple(encoded_tup, merge_idx):
    # merge all the (idx, idx+1) for idx in merge_idx 
    # edge case, idx, idx+1 both in merge_idx, only merge idx, remove idx+1 (skip)
    tmp = encoded_tup

    merged = 0
    prev = None
    for i in merge_idx:
        if prev and prev == i - 1: # if idx, idx+1 already merged, no more merge for
            # idx+1 and idx+2
            continue
        tmp = merge_single_idx(tmp, i - merged)
        prev = i
        merged += 1
    return tmp


In [31]:
from pathlib import Path
FIXTURES_PATH = Path('tests/fixtures')
VOCAB_PATH = FIXTURES_PATH / "gpt2_vocab.json"
MERGES_PATH = FIXTURES_PATH / "gpt2_merges.txt"
from __future__ import annotations

import pathlib
from functools import lru_cache

# FIXTURES_PATH = (pathlib.Path(__file__).resolve().parent) / "fixtures"
import json


@lru_cache
def gpt2_bytes_to_unicode() -> dict[int, str]:
    """
    Returns a mapping between every possible byte (an integer from 0 to 255) to a
    printable unicode string character representation. This function is taken
    from the GPT-2 code.

    For example, `chr(0)` is `\x00`, which is an unprintable character:

    >>> chr(0)
    '\x00'
    >>> print(chr(0))

    As a result, this function returns a dictionary `d` where `d[0]` returns `Ā`.
    The bytes that are visually printable keep their original string representation [1].
    For example, `chr(33)` returns `!`, and so accordingly `d[33]` returns `!`.
    Note in particular that the space character `chr(32)` becomes `d[32]`, which
    returns 'Ġ'.

    For unprintable characters, the function shifts takes the integer representing
    the Unicode code point of that character (returned by the Python `ord`) function
    and shifts it by 256. For example, `ord(" ")` returns `32`, so the the space character
    ' ' is shifted to `256 + 32`. Since `chr(256 + 32)` returns `Ġ`, we use that as the
    string representation of the space.

    This function can simplify the BPE implementation and makes it slightly easier to
    manually inspect the generated merges after they're serialized to a file.
    """
    # These 188 integers can used as-is, since they are not whitespace or control characters.
    # See https://www.ssec.wisc.edu/~tomw/java/unicode.html.
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    # now get the representations of the other 68 integers that do need shifting
    # each will get mapped chr(256 + n), where n will grow from 0...67 in the loop
    # Get printable representations of the remaining integers 68 integers.
    n = 0
    for b in range(2**8):
        if b not in bs:
            # If this integer isn't in our list of visually-representable
            # charcters, then map it to the next nice character (offset by 256)
            bs.append(b)
            cs.append(2**8 + n)
            n += 1
    characters = [chr(n) for n in cs]
    d = dict(zip(bs, characters))
    return d

def get_tokenizer_from_vocab_merges_path(
    vocab_path: str | os.PathLike,
    merges_path: str | os.PathLike,
    special_tokens: list[str] | None = None,
):
    gpt2_byte_decoder = {v: k for k, v in gpt2_bytes_to_unicode().items()}
    with open(vocab_path) as vocab_f:
        gpt2_vocab = json.load(vocab_f)
    gpt2_bpe_merges = []
    with open(merges_path) as f:
        for line in f:
            cleaned_line = line.rstrip()
            if cleaned_line and len(cleaned_line.split(" ")) == 2:
                gpt2_bpe_merges.append(tuple(cleaned_line.split(" ")))
    # The GPT-2 tokenizer uses a remapped unicode encoding for bytes. Let's
    # just return the original bytes, so we don't force students to use
    # any particular encoding scheme.
    vocab = {
        gpt2_vocab_index: bytes([gpt2_byte_decoder[token] for token in gpt2_vocab_item])
        for gpt2_vocab_item, gpt2_vocab_index in gpt2_vocab.items()
    }
    # If any of the special tokens don't exist in the vocab, append them to the vocab.
    if special_tokens:
        for special_token in special_tokens:
            byte_encoded_special_token = special_token.encode("utf-8")
            if byte_encoded_special_token not in set(vocab.values()):
                vocab[len(vocab)] = byte_encoded_special_token

    merges = [
        (
            bytes([gpt2_byte_decoder[token] for token in merge_token_1]),
            bytes([gpt2_byte_decoder[token] for token in merge_token_2]),
        )
        for merge_token_1, merge_token_2 in gpt2_bpe_merges
    ]
    return get_tokenizer(vocab, merges, special_tokens)

def get_tokenizer(
    vocab: dict[int, bytes],
    merges: list[tuple[bytes, bytes]],
    special_tokens: list[str] | None = None,
):
    """Given a vocabulary, a list of merges, and a list of special tokens,
    return a BPE tokenizer that uses the provided vocab, merges, and special tokens.

    Args:
        vocab (dict[int, bytes]): The tokenizer vocabulary, a mapping from int (token ID in the vocabulary)
            to bytes (token bytes)
        merges (list[tuple[bytes, bytes]]): BPE merges. Each list item is a tuple of bytes (<token1>, <token2>),
            representing that <token1> was merged with <token2>.
            Merges are ordered by order of creation.
        special_tokens (list[str] | None): A list of string special tokens for the tokenizer. These strings will never
            be split into multiple tokens, and will always be kept as a single token.

    Returns:
        A BPE tokenizer that uses the provided vocab, merges, and special tokens.
    """
    # raise NotImplementedError

    return Tokenizer(vocab, merges, special_tokens=special_tokens)


In [32]:
# class Tokenizer:
#     def __init__(
#             self, 
#             vocab: dict[int, bytes], 
#             merges: list[tuple[bytes, bytes]], 
#             special_tokens:list[str]|None=None
#     ) -> None:
#         self.vocab = vocab
#         self.merges = merges
#         self.special_tokens = special_tokens
#         self.special_ids = set()
#         self.merges_rank = {merges[i]:i for i in range(len(merges))}
#         self.PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
#         for token in self.special_tokens:
#             idx = len(self.vocab)
#             self.vocab[idx] = token
#             self.special_ids.add(idx)
#         self.inv_vocab = {v:k for k, v in self.vocab.items()}
#         print(special_tokens[0] in self.inv_vocab)
#     @classmethod
#     def from_files(
#         cls, 
#         vocab_filepath,
#         merges_filepath,
#         special_tokens
#     ):
#         vocab = read_pkl(vocab_filepath)
#         merges = read_pkl(merges_filepath)
#         return Tokenizer(vocab, merges, special_tokens)

#     def encode(
#             self,
#             text: str
#     ):
#         output_encoded_lis = []
#         text_lis = split_special(text, self.special_tokens, keep_split=True)
#         for i in range(len(text_lis)):
            
#             if i%2 == 0:
#                 cur_lis = []
#                 doc_split = re.findall(self.PAT, text_lis[i])
#                 for j in range(len(doc_split)):
#                     encoded_tup = string_to_byte_tup(doc_split[j])
#                     while True:
#                         min_pair, min_rank, merge_idx = get_merge_pair(encoded_tup, self.merges_rank)
#                         if min_pair is None:
#                             break
#                         encoded_tup = merge_tuple(encoded_tup, merge_idx)
#                     for token in encoded_tup:
#                         # cur_lis.append(self.inv_vocab[token])
#                         output_encoded_lis.append(self.inv_vocab[token])
#                     # output_encoded_lis.append(cur_lis)
#             else:
#                 # get the id for special tokens directly from vocab
#                 print(text_lis[i])
#                 output_encoded_lis.append(self.inv_vocab[text_lis[i]]) 
#         return output_encoded_lis
                    
#     def encode_iterable(
#         self, 
#         iterable
#     ):
#         for item in iterable:
#             yield from self.encode(item)

#     def decode(
#             self,
#             ids
#     ):
#         output_lis = []
#         cur_ = []
#         for i in range(len(ids)):
#             token = self.vocab[ids[i]] 
#             # regular token, append to the crrent processing tokens
#             if token not in self.special_tokens:
#                 cur_.append(token)
#             if token in self.special_tokens or i == len(ids)-1: # stop condition -> conver the cur_ processing regular byte tokens to string
#                 # print(cur_)
#                 cur_str = b''.join(cur_).decode('utf-8', errors="replace")
#                 output_lis.append(cur_str) # output these current processing tokens
#                 cur_ = []
#             # if the stop conditoin is a special token, then append this special token to the output lis
#                 if token in self.special_tokens:
#                     output_lis.append(token)
#         return ''.join(output_lis)
            
                


        

In [33]:
class Tokenizer:
    def __init__(
            self, 
            vocab: dict[int, bytes], 
            merges: list[tuple[bytes, bytes]], 
            special_tokens:list[str]|None=None
    ) -> None:
        self.vocab = vocab
        self.inv_vocab = {v:k for k, v in vocab.items()}
        self.merges = merges
        self.special_tokens = special_tokens
        # self.b_special_tokens =  [tok.encode('utf-8') for tok in special_tokens]
        self.merges_rank = {merges[i]:i for i in range(len(merges))}
        self.PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
        if self.special_tokens is not None:
            for token in special_tokens:
                b_token = token.encode('utf-8')
                # print(b_token, "in vocab?", b_token in self.inv_vocab)
                if b_token not in self.inv_vocab:
                    ls = len(self.vocab)
                    self.vocab[ls] = b_token
                    self.inv_vocab[b_token] = ls

    @classmethod
    def from_files(
        cls, 
        vocab_filepath,
        merges_filepath,
        special_tokens
    ):
        vocab = read_pkl(vocab_filepath)
        merges = read_pkl(merges_filepath)
        return Tokenizer(vocab, merges, special_tokens)

    def encode(
            self,
            text: str
    ):
        output_encoded_lis = []
        text_lis = split_special(text, self.special_tokens, keep_split=True)
        for i in range(len(text_lis)):
            if i%2 == 0:
                doc_split = re.findall(self.PAT, text_lis[i])
                for j in range(len(doc_split)):
                    encoded_tup = string_to_byte_tup(doc_split[j])
                    while True:
                        min_pair, min_rank, merge_idx = get_merge_pair(encoded_tup, self.merges_rank)
                        # print(min_pair, min_rank, merge_idx)
                        if min_pair is None:
                            break

                        encoded_tup = merge_tuple(encoded_tup, merge_idx)

                    for token in encoded_tup:
                        output_encoded_lis.append(self.inv_vocab[token])
            else:
                # get the id for special tokens directly from vocab
                output_encoded_lis.append(self.inv_vocab[text_lis[i].encode('utf-8', errors = 'replace')]) 
        return output_encoded_lis
                    
    def encode_iterable(
        self, 
        iterable
    ):
        for item in iterable:
            yield from self.encode(item)
    def decode(
            self,
            ids
    ):
        return (b''.join(self.vocab[i] for i in ids)).decode('utf-8', errors = 'replace')
        

In [34]:
special_tokens=["<|endoftext|>"]
test_string = "Héllò hôw <|endoftext|><|endoftext|> are ü? 🙃<|endoftext|>"
def test_roundtrip_unicode_string_with_special_tokens():
    tokenizer = get_tokenizer_from_vocab_merges_path(
        vocab_path=VOCAB_PATH, merges_path=MERGES_PATH, special_tokens=["<|endoftext|>"]
    )
    test_string = "Héllò hôw <|endoftext|><|endoftext|> are ü? 🙃<|endoftext|>"
    encoded_ids = tokenizer.encode(test_string)
    # print(encoded_ids)
    tokenized_string = [tokenizer.decode([x]) for x in encoded_ids]
    # print(tokenized_string)
    # Ensure the special <|endoftext|> token is preserved
    assert tokenized_string.count("<|endoftext|>") == 3

    decoded_string = tokenizer.decode(encoded_ids)
    assert test_string == decoded_string

In [35]:
import tiktoken
def test_unicode_string_with_special_tokens_matches_tiktoken():
    reference_tokenizer = tiktoken.get_encoding("gpt2")
    tokenizer = get_tokenizer_from_vocab_merges_path(
        vocab_path=VOCAB_PATH, merges_path=MERGES_PATH, special_tokens=["<|endoftext|>"]
    )
    test_string = "Héllò hôw <|endoftext|><|endoftext|> are ü? 🙃<|endoftext|>"

    reference_ids = reference_tokenizer.encode(test_string, allowed_special={"<|endoftext|>"})
    ids = tokenizer.encode(test_string)
    print(ids)
    print(reference_ids)
    assert ids == reference_ids

    assert tokenizer.decode(ids) == test_string
    assert reference_tokenizer.decode(reference_ids) == test_string


In [36]:
def test_address_roundtrip():
    tokenizer = get_tokenizer_from_vocab_merges_path(
        vocab_path=VOCAB_PATH,
        merges_path=MERGES_PATH,
    )
    with open(FIXTURES_PATH / "address.txt") as f:
        corpus_contents = f.read()
    print(corpus_contents[:10])
    ids = tokenizer.encode(corpus_contents)
    assert tokenizer.decode(ids) == corpus_contents

In [37]:
def test_german_matches_tiktoken():
    reference_tokenizer = tiktoken.get_encoding("gpt2")
    tokenizer = get_tokenizer_from_vocab_merges_path(
        vocab_path=VOCAB_PATH,
        merges_path=MERGES_PATH,
    )
    corpus_path = FIXTURES_PATH / "german.txt"
    with open(corpus_path) as f:
        corpus_contents = f.read()
    reference_ids = reference_tokenizer.encode(corpus_contents)
    ids = tokenizer.encode(corpus_contents)
    assert ids == reference_ids

    assert tokenizer.decode(ids) == corpus_contents
    assert reference_tokenizer.decode(reference_ids) == corpus_contents

In [40]:
# tokenizer = Tokenizer(vocab = vocab, merges = merges)
# print(vocab)
test_roundtrip_unicode_string_with_special_tokens()
test_unicode_string_with_special_tokens_matches_tiktoken()
test_address_roundtrip()
test_german_matches_tiktoken()


[39, 2634, 297, 127, 110, 289, 27083, 86, 220, 50256, 50256, 389, 6184, 120, 30, 12520, 247, 225, 50256]
[39, 2634, 297, 127, 110, 289, 27083, 86, 220, 50256, 50256, 389, 6184, 120, 30, 12520, 247, 225, 50256]
Four score


## Experiments with tokenizers

In [16]:
from train_bpe import read_pkl, save_pkl, Tokenizer
vocab_tiny_story = 'cs336_basics/test/vocab_tiny_story.pkl'
merges_tiny_story = 'cs336_basics/test/merges_tiny_story.pkl'
# vocab = read_pkl(vocab_tiny_story)
# merges = read_pkl(merges_tiny_story)

In [17]:
print(vocab)
print(merges)

[output cleared: 342 KB]


In [18]:
tokenizer = Tokenizer.from_files(
    vocab_filepath=vocab_tiny_story,
    merges_filepath=merges_tiny_story,
    special_tokens=["<|endoftext|>"]
)

In [ ]:
sample_path = 'tests/fixtures/tinystories_sample.txt'
with open(sample_path, 'r') as f:
    text = f.read()
ids = tokenizer.encode(text)
len(ids)/len(text)


0.23560486001056524